# Experiment 1: Comparing pretrained models and skill aliases

**Question:** Can we improve the zero-shot cosine baseline without training by
1. using another pretrained embedding model (`all-mpnet-base-v2` or the domain-specific `TechWolf/JobBERT-v2`); and
2. representing each skill with **all of its aliases**, not just the first one?

**Skill alias strategies** (each job title in the dev set has only one form, so aliases apply only to skills):
- `first` — use only `skill_aliases[0]`, as in the baseline;
- `mean` — average the embeddings of all aliases;
- `max` — use the highest cosine similarity across a skill's aliases.

We use the same metrics as the baseline (`ranking_metrics.py`) and evaluate on the full dev corpus.

In [1]:
import os, sys
from pathlib import Path

# The notebook is in experiments/, while config.py and ranking_metrics.py are in the project root.
if not Path("config.py").exists():
    os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /Users/mac/spj/project/Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches


In [2]:
import ast
import json
import gc

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from ranking_metrics import evaluate_query, save_results

VAL_DIR = Path("data/raw/TaskB/validation")

# Short name (used in output filenames) -> Hugging Face model name
MODELS = {
    "minilm": "all-MiniLM-L6-v2",
    "mpnet": "all-mpnet-base-v2",
    "jobbert": "TechWolf/JobBERT-v2",
}
ALIAS_MODES = ["first", "mean", "max"]

## 1. Data (same as the baseline)

In [3]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t").rename(columns={"jobtitle": "job_name", "q_id": "job_id"})
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t").rename(columns={"c_id": "skill_id"})
corpus["skill_aliases"] = corpus["skill_aliases"].apply(ast.literal_eval)
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", header=None, sep="\t", names=["job_id", "_", "skill_id", "label"])

skill_index = {sid: i for i, sid in enumerate(corpus["skill_id"])}
grades_by_job = {}
for r in qrels.itertuples(index=False):
    grades_by_job.setdefault(r.job_id, np.zeros(len(corpus)))[skill_index[r.skill_id]] = r.label

# Flatten all aliases; starts[i] is the index of the first alias for skill i.
alias_counts = corpus["skill_aliases"].apply(len).values
assert alias_counts.min() >= 1
alias_texts = [a for aliases in corpus["skill_aliases"] for a in aliases]
starts = np.r_[0, np.cumsum(alias_counts)[:-1]]
owner = np.repeat(np.arange(len(corpus)), alias_counts)  # Skill index for each alias

print(f"queries: {len(queries)}, skills: {len(corpus)}, aliases: {len(alias_texts)} "
      f"(average {alias_counts.mean():.1f} per skill)")

queries: 304, skills: 9052, aliases: 64321 (average 7.1 per skill)


## 2. Experiment
For each model, we encode the job titles and all unique alias texts, then evaluate the three strategies.
If a model cannot be loaded, the notebook reports the error and moves on to the next one.

In [4]:
def encode_texts(model, texts):
    unique = sorted(set(texts))
    vecs = model.encode(unique, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    return dict(zip(unique, vecs))


def evaluate(score_fn, job_emb):
    per_query = []
    for row in queries.itertuples(index=False):
        if row.job_id not in grades_by_job:
            continue
        per_query.append(evaluate_query(score_fn(job_emb[row.job_name]), grades_by_job[row.job_id]))
    return per_query


for short, model_name in MODELS.items():
    print(f"\n===== {model_name} =====")
    try:
        model = SentenceTransformer(model_name)
        job_emb = encode_texts(model, queries["job_name"].tolist())
        alias_map = encode_texts(model, alias_texts)
    except Exception as e:  # The model is unavailable or uses an incompatible interface
        print(f"Skipping {model_name}: {type(e).__name__}: {e}")
        continue

    A = np.vstack([alias_map[t] for t in alias_texts])  # (number of aliases, dim)

    # mean: average each skill's alias vectors, then normalize to unit length
    A_mean = np.zeros((len(corpus), A.shape[1]), dtype=np.float32)
    np.add.at(A_mean, owner, A)
    A_mean /= np.linalg.norm(A_mean, axis=1, keepdims=True)

    scorers = {
        "first": lambda e: A[starts] @ e,
        "mean": lambda e: A_mean @ e,
        "max": lambda e: np.maximum.reduceat(A @ e, starts),
    }
    descriptions = {
        "first": f"Zero-shot cosine ({model_name}); each skill is represented by its first alias.",
        "mean": f"Zero-shot cosine ({model_name}); each skill is represented by the mean of its alias embeddings.",
        "max": f"Zero-shot cosine ({model_name}); each skill's score is the highest cosine similarity across its aliases.",
    }
    for mode in ALIAS_MODES:
        per_query = evaluate(scorers[mode], job_emb)
        save_results(f"cosine_{short}_{mode}", descriptions[mode], per_query)

    del model, A, A_mean, alias_map
    gc.collect()


===== all-MiniLM-L6-v2 =====


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/986 [00:00<?, ?it/s]

Збережено: results/cosine_minilm_first.json
Збережено: results/cosine_minilm_mean.json
Збережено: results/cosine_minilm_max.json

===== all-mpnet-base-v2 =====


...

Збережено: results/cosine_mpnet_first.json
Збережено: results/cosine_mpnet_mean.json
Збережено: results/cosine_mpnet_max.json

===== TechWolf/JobBERT-v2 =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/986 [00:00<?, ?it/s]

Збережено: results/cosine_jobbert_first.json
Збережено: results/cosine_jobbert_mean.json
Збережено: results/cosine_jobbert_max.json


## 3. Comparison
As a sanity check, `cosine_minilm_first` should match `cosine_minilm` from the baseline notebook (graded nDCG ≈ 0.615).

In [6]:
rows = []
for p in sorted(Path("results").glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"approach": p.stem, **{k: v for k, v in d.items() if k != "description"}})

pd.DataFrame(rows).sort_values("ndcg_graded", ascending=False).reset_index(drop=True)

AttributeError: 'list' object has no attribute 'items'